# DotX Solar Forecast API - Quickstart Notebook

End-to-end walkthrough of the Solar Forecast API in raw HTTP, with inline plots.

You will:

1. Register a plant
2. Register a solar asset
3. Calibrate the asset model using cumulative-energy readings from your CSV
4. Poll until calibration completes, then plot measured vs. modeled
5. Request a forecast and plot the forecast curve

All calls use a single credential: your tenant `X-API-Key`.

**Prerequisites in your `.env` file:**

- `EMS_API_KEY` - your tenant key
- `EMS_API_BASE` - the API base URL (your DotX contact will provide it)

Edit the **CONFIGURATION** cell below (`CSV_PATH`, plant name, latitude, longitude, asset config) to match your plant before running. `CSV_PATH` points at a CSV with columns `timestamp,solar` (cumulative Wh; the inverter's running watt-hour counter) at 15-minute resolution.

In [ ]:
import csv
import os
import time
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import requests
from dotenv import load_dotenv

%matplotlib inline

load_dotenv()

API_BASE = os.environ['EMS_API_BASE'].rstrip('/')
API_KEY  = os.environ['EMS_API_KEY']

# --- CONFIGURATION (edit for your plant) ---
CSV_PATH        = r"C:\path\to\your\csv.csv"   # CSV: timestamp,solar (cumulative Wh, 15-min, >=97 rows)

PLANT_NAME    = 'My Plant'
LATITUDE      = 52.0
LONGITUDE     = 4.20
TIMEZONE      = 'Europe/Amsterdam'

ASSET_NAME      = 'Solar asset'
INVERTER_AC_KW  = 100
EFFICIENCY      = 0.9
TEMP_COEFF      = -0.0029   # power loss per °C above 25 °C cell temp [1/°C]; typical c-Si ≈ -0.0029
DC_KWP          = 94.0        # None -> computed from SUB_ARRAYS, or pin to a number
SUB_ARRAYS      = [
    {'name': 'main', 'kwp': None, 'tilt': None, 'azimuth': None},
]

headers = {'X-API-Key': API_KEY, 'Content-Type': 'application/json'}


def http(method, url, *, json=None, params=None, expect=None):
    r = requests.request(method, url, headers=headers, json=json, params=params, timeout=60)
    if (expect is not None and r.status_code not in expect) or (expect is None and not r.ok):
        print(f'! HTTP {r.status_code}: {r.text}')
        r.raise_for_status()
    return r


print(f'API_BASE = {API_BASE}')
print(f'CSV file: {os.path.basename(CSV_PATH)}')

## Step 1 - Register a plant

`POST /plants/` registers a plant under your tenant. The plant `name` is unique per tenant — a new name creates a new plant (`201`), a duplicate name returns `409 Conflict`. All calls authenticate with the tenant `X-API-Key`; no key is minted at this step.

In [ ]:
plant_payload = {
    'name':      PLANT_NAME,
    'latitude':  LATITUDE,
    'longitude': LONGITUDE,
    'timezone':  TIMEZONE,
}

response = requests.post(f'{API_BASE}/plants/', headers=headers, json=plant_payload, timeout=60)
if response.status_code == 409:
    raise RuntimeError(
        f"Plant with name '{PLANT_NAME}' already exists in this tenant. "
        f"Change PLANT_NAME in the configuration cell to create a new plant."
    )
response.raise_for_status()
plant_id = response.json()['plant_id']
print(f'Created plant {plant_id}')

## Step 2 - Register a solar asset

`POST /plants/{plant_id}/assets/` describes the physical configuration: sub-arrays (each with `kwp` / `tilt` / `azimuth`), inverter AC capacity (`inverter_ac_kw`), DC-side `efficiency`, and panel temperature coefficient (`temp_coeff`, in 1/°C - power loss per °C above 25 °C cell temperature). Any of `kwp` / `tilt` / `azimuth` can be left `null` - the autofit will determine them.

In [ ]:
asset_payload = {
    'name':           ASSET_NAME,
    'inverter_ac_kw': INVERTER_AC_KW,
    'efficiency':     EFFICIENCY,
    'temp_coeff':     TEMP_COEFF,
    'sub_arrays':     SUB_ARRAYS,
}
if DC_KWP is not None:
    asset_payload['dc_kwp'] = DC_KWP

response = http('POST', f'{API_BASE}/plants/{plant_id}/assets/', json=asset_payload, expect=(201,))
asset_id = response.json()['asset_id']
print(f'asset_id = {asset_id}')

## Step 3 - Calibrate the asset model

`POST /plants/{plant_id}/assets/{asset_id}/fit/` accepts `{"measurements": [{"time": ISO8601, "solar": int}, ...]}` where `solar` is the inverter's cumulative energy counter in **watt-hours** (the monotonic lifetime-yield reading, not instantaneous power). Submit **at least 97 points** (one day at 15-min resolution: 96 power values need 97 cumulative samples to diff). The fit runs as an async task on our side and returns a `task_id` immediately.

The cell below loads your CSV (`timestamp,solar` columns), converts to the JSON shape, and dispatches the fit.

In [ ]:
def load_measurements(path):
    out = []
    with open(path, newline='') as f:
        for row in csv.DictReader(f):
            ts = datetime.fromisoformat(row['timestamp'].replace(' ', 'T').replace('Z', '+00:00'))
            if ts.tzinfo is None:
                ts = ts.replace(tzinfo=timezone.utc)
            out.append({
                'time':  ts.astimezone(timezone.utc).isoformat().replace('+00:00', 'Z'),
                'solar': int(row['solar']),
            })
    return out


measurements = load_measurements(CSV_PATH)
print(f'Loaded {len(measurements)} measurements ({measurements[0]["time"]} -> {measurements[-1]["time"]})')

response = http('POST', f'{API_BASE}/plants/{plant_id}/assets/{asset_id}/fit/',
                json={'measurements': measurements}, expect=(202,))
task = response.json()
print(f"Fit task dispatched: {task['task_id']} (status={task['status']})")

## Step 4 - Poll for calibration, plot measured vs. modeled

`GET /fit/` has three modes depending on query parameters:

- **No params**: returns `{is_calibrated, fitted_params, status}` — the current calibration snapshot for the asset (not a task progress indicator).
- **`?start=ISO&end=ISO`**: dispatches an async task that computes the modeled power curve over the requested window. Returns `202 {task_id}`.
- **`?task_id=...`**: polls a previously-dispatched task (autofit OR modeled-timeseries). `202 PENDING` while running, `200` with the task result when done.

Use the `task_id` returned by the `POST /fit/` call in Step 3 and poll `?task_id=` until the response is `200`; the body carries `r2` and `rmse`.

In [ ]:
fit_url = f'{API_BASE}/plants/{plant_id}/assets/{asset_id}/fit/'
fit_task_id = task['task_id']
start_t = time.monotonic()

while True:
    if time.monotonic() - start_t > 600:
        raise RuntimeError('Fit did not complete within 600s')
    r = requests.get(fit_url, headers=headers, params={'task_id': fit_task_id}, timeout=60)
    if r.status_code == 200:
        fit_result = r.json()
        break
    if r.status_code != 202:
        print(f'! HTTP {r.status_code}: {r.text}')
        r.raise_for_status()
    print(f"... task still running ({time.monotonic() - start_t:.0f}s elapsed)")
    time.sleep(5)

fitted_params = fit_result['fitted_params']
r2   = fitted_params['r2']
rmse = fitted_params['rmse']
print(f'Calibrated. R² = {r2:.3f}, RMSE = {rmse:.2f} kW')

print(f"Fitted DC capacity: {fitted_params['fitted_kwp']:.2f} kWp")
for sa in fitted_params['sub_arrays']:
    print(
        f"  {sa['name']}: "
        f"kwp = {sa['kwp']:.2f} kWp, "
        f"tilt = {sa['tilt_deg']:.1f}°, "
        f"azimuth = {sa['azimuth_deg']:.1f}°"
    )

In [ ]:
# Dispatch the modeled-timeseries computation over the same window as the measurements.
fit_start_iso = measurements[0]['time']
fit_end_iso   = measurements[-1]['time']

ts_response = http(
    'GET', fit_url,
    params={'start': fit_start_iso, 'end': fit_end_iso},
    expect=(202,),
)
ts_task_id = ts_response.json()['task_id']
print(f'Modeled-timeseries task: {ts_task_id}')

ts_start = time.monotonic()
while True:
    if time.monotonic() - ts_start > 600:
        raise RuntimeError('Modeled timeseries did not arrive within 600s')
    r = requests.get(fit_url, headers=headers, params={'task_id': ts_task_id}, timeout=60)
    if r.status_code == 200:
        modeled = r.json()
        break
    elif r.status_code == 202:
        time.sleep(2)
    else:
        print(f'! HTTP {r.status_code}: {r.text}')
        r.raise_for_status()

print(f"Got {len(modeled['samples'])} modeled samples.")

In [ ]:
from collections import defaultdict
from statistics import mean

# Diff cumulative Wh -> average kW: (solar[i+1]-solar[i]) Wh over 0.25 h = delta/250 kW.
# Tag each derived value with midpoint(t_i, t_{i+1}) + 15 min, matching the server's convention.
measured_kw    = [min((measurements[i+1]['solar'] - measurements[i]['solar']) / 250, 1.2 * DC_KWP) for i in range(len(measurements) - 1)]
measured_times = [
    datetime.fromisoformat(measurements[i+1]['time'].replace('Z', '+00:00')) + timedelta(minutes=7.5)
    for i in range(len(measurements) - 1)
]
modeled_times  = [datetime.fromisoformat(s['time'].replace('Z', '+00:00')) for s in modeled['samples']]
modeled_kw_arr = [s['modeled_kw'] for s in modeled['samples']]

DT_HOURS = 0.25  # 15-min sample spacing

# Daily kWh per series (bucket each by date independently - robust to small
# timestamp mismatches between the customer CSV and the modeled samples).
meas_daily = defaultdict(float)
for t, kw in zip(measured_times, measured_kw):
    meas_daily[t.date()] += kw * DT_HOURS
mod_daily = defaultdict(float)
for t, kw in zip(modeled_times, modeled_kw_arr):
    mod_daily[t.date()] += kw * DT_HOURS

meas_days  = sorted(meas_daily)
mod_days   = sorted(mod_daily)
meas_kwh_d = [meas_daily[d] for d in meas_days]
mod_kwh_d  = [mod_daily[d]  for d in mod_days]

# MAE over days present in both series (drop nights / <1 kWh days).
common_days = [d for d in (set(meas_daily) & set(mod_daily)) if meas_daily[d] >= 1.0]
errs        = [mod_daily[d] - meas_daily[d] for d in common_days]
mae_kwh     = mean(abs(e) for e in errs) if errs else 0.0
mean_act    = mean(meas_daily[d] for d in common_days) if common_days else 0.0
nmae_pct    = (mae_kwh / mean_act * 100) if mean_act else 0.0

# Cumulative kWh per series independently.
def _cumulative(values, dt):
    out, total = [], 0.0
    for v in values:
        total += v * dt
        out.append(total)
    return out

cum_meas       = _cumulative(measured_kw,    DT_HOURS)
cum_mod        = _cumulative(modeled_kw_arr, DT_HOURS)
total_meas_kwh = cum_meas[-1] if cum_meas else 0.0
total_mod_kwh  = cum_mod[-1]  if cum_mod  else 0.0
energy_err_pct = ((total_mod_kwh - total_meas_kwh) / total_meas_kwh * 100) if total_meas_kwh else 0.0

print(f'Measured: {len(measured_times)} samples, {len(meas_days)} days  |  '
      f'Modeled: {len(modeled_times)} samples, {len(mod_days)} days  |  '
      f'Common days for MAE: {len(common_days)}')

fig, axes = plt.subplots(3, 1, figsize=(12, 10), constrained_layout=True)

axes[0].plot(measured_times, measured_kw,    label='Measured', linewidth=0.6, color='#808080')
axes[0].plot(modeled_times,  modeled_kw_arr, label='Modeled',  linewidth=0.6, color='#FFB700')
axes[0].set_ylabel('Power (kW)')
axes[0].set_title(f'Measured vs. modeled - R² = {r2:.3f}, RMSE = {rmse:.2f} kW')
axes[0].legend(loc='upper left')
axes[0].grid(alpha=0.3)

axes[1].plot(meas_days, meas_kwh_d, 'o-', label='Measured', linewidth=1.5, markersize=4, color='#808080')
axes[1].plot(mod_days,  mod_kwh_d,  'o-', label='Modeled',  linewidth=1.5, markersize=4, color='#FFB700')
axes[1].set_ylabel('kWh / day')
axes[1].set_title(f'Daily yield  |  MAE: {mae_kwh:,.0f} kWh/day ({nmae_pct:.1f} %)')
axes[1].legend(loc='upper left')
axes[1].grid(alpha=0.3)

axes[2].plot(measured_times, cum_meas, label='Measured', linewidth=1.5, color='#808080')
axes[2].plot(modeled_times,  cum_mod,  label='Modeled',  linewidth=1.5, color='#FFB700')
axes[2].set_xlabel('Time (UTC)')
axes[2].set_ylabel('kWh')
axes[2].set_title(
    f'Cumulative yield  |  measured: {total_meas_kwh:,.0f} kWh  '
    f'modeled: {total_mod_kwh:,.0f} kWh  (error: {energy_err_pct:+.1f} %)'
)
axes[2].legend(loc='upper left')
axes[2].grid(alpha=0.3)

fig.autofmt_xdate()
plt.show()

## Step 5 - Request a forecast

`POST /plants/{plant_id}/assets/{asset_id}/forecast/` is the only way to retrieve a forecast: you must supply a handful of recent measurements alongside the request. The API stores those readings and returns a fresh forecast (typically 48 hours ahead).

In [ ]:
# /forecast/ still expects instantaneous power_kw, so derive two readings by diffing
# the last three cumulative samples: (Wh delta over 15 min) / 250 = average kW.
s0, s1, s2 = measurements[-3]['solar'], measurements[-2]['solar'], measurements[-1]['solar']
recent_pk1 = (s1 - s0) / 250
recent_pk2 = (s2 - s1) / 250

recent_now = datetime.now(timezone.utc).replace(second=0, microsecond=0)
recent_measurements = [
    {'time': (recent_now - timedelta(minutes=30)).isoformat().replace('+00:00', 'Z'), 'power_kw': recent_pk1},
    {'time': (recent_now - timedelta(minutes=15)).isoformat().replace('+00:00', 'Z'), 'power_kw': recent_pk2},
]

response = http('POST', f'{API_BASE}/plants/{plant_id}/assets/{asset_id}/forecast/',
                json={'measurements': recent_measurements}, expect=(200,))
forecast = response.json()

print(f"Performance ratio : {forecast['performance_ratio']:.3f}")
print(f"Horizon           : {forecast['horizon_hours']} hours, {len(forecast['forecast_data'])} samples")

In [ ]:
fc_times = [datetime.fromisoformat(s['time'].replace('Z', '+00:00')) for s in forecast['forecast_data']]
fc_kw    = [s['power_kw'] for s in forecast['forecast_data']]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(fc_times, fc_kw, linewidth=1.0)
ax.set_xlabel('Time (UTC)')
ax.set_ylabel('Forecast power (kW)')
ax.set_title(f"{forecast['horizon_hours']}-hour forecast")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()